# Обучение модели

Устанавливаем наш пакет и импортируем все нужное

In [ ]:
%pip install -q git+https://github.com/tereshkovyv/transformer_lab.git

Подключаем Гугл диск, чтобы хранить там временные файлы

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/transformer_lab/data/wikitext-103"

Настраиваем конфигурацию

In [ ]:
SEED = 44
cfg = {"vocab_size": 32000, "context_length": 256, "emb_dim": 768, "n_heads": 12,
       "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False}
hp = {"tokenizer": "llama2", "batch_size": 8, "lr": 5e-4, "weight_decay": 0.1, "num_epochs": 1,
      "eval_freq": 500, "eval_iter": 20}

Создаем объекты данных, модели, оптимайзера

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformer_lab.model.gpt_model import GPTModel
from transformer_lab.training.data import TokenDataset

device = "cuda"
torch.manual_seed(SEED)

train_ds = TokenDataset("train", cfg["context_length"], DATA_DIR, hp["tokenizer"])
val_ds = TokenDataset("validation", cfg["context_length"], DATA_DIR, hp["tokenizer"])
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=hp["batch_size"], shuffle=True, drop_last=True, generator=g)
val_loader = DataLoader(val_ds, batch_size=hp["batch_size"], shuffle=False, drop_last=True)

model = GPTModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])

Обучаем

In [ ]:
from transformer_lab.training.data import Llama2Tokenizer
from transformer_lab.training.train import train_model_simple
train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=hp["num_epochs"], eval_freq=hp["eval_freq"], eval_iter=hp["eval_iter"],
    start_context="The history of", tokenizer=Llama2Tokenizer(DATA_DIR))

Сохраняем веса

In [ ]:
from google.colab import files
torch.save(model.state_dict(), "model.pth")
files.download("model.pth")